# 02 · Count audit — Fetal death, MIMIC-IV (M3)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 · G1 → **02: S4** → 03: S5 · S6 → R3 → 04: R3 · S2 · S3 · G1 · S4 → 05: S5 · S6 · S7 · G2 → 06: G3 · class · S11 |
| What it does | Runs the count audit of groups A to D (base, events and definition, patient overlap, era and coding) and shows the decisions that each group settled before any model |
| Stages and gates | S4 (it settles D1, D3, D4 and D5; it prepares S5, S6 and criterion C4) |
| Decisions | D1 (database), D3 (population), D4 (unit), D5 (outcome); see `count_audit.md`, section 4 |
| Inputs | Revised obstetric cohort (`cohort_obstetric_v2`, `hadm_delivery_v2`, `obstetric_dx_v2`, `predictors_24h_v2`) |
| Script | `sql/02_audit_counts/audit_counts.sql` (read-only: temporary tables and SELECT) |
| Outputs | `data/outputs_02_count_audit.txt` (aggregate counts only; 1 to 4 shown as "<5") |
| Time | One to two minutes |
| Previous / next | `01_outcome_definition` / `03_unit_and_prediction_point` |
| Spanish version | `02_count_audit_es.ipynb` (same code) |

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
engine = get_connect_database()

## S4 · Count audit, group A — base counts and reconciliation (decision D1)

The audit measures, before any model, the numbers that decide the design. It works on the revised obstetric cohort and follows two rules: **no overlap and no selection by outcome**. Group A checks that the base counts match the maternal audit and that the label reproduces the earlier fetal-death dataset (Q-A2 needs the earlier table `cohort_fetal_death_dataset`; if it does not exist, only that query is reported as failed).

| Result | What is done | In this model |
|---|---|---|
| Base counts match and the label reproduces the earlier dataset | D1: use the revised tables of the obstetric cohort | 25,736 admissions; 154 events in both; 0 discordant |
| They differ | Review the cohort before going on | — |

In [ ]:
# Stage S4: count audit, groups A to D, in one connection (read-only script)
aud = run_script(engine, SQL / "02_audit_counts" / "audit_counts.sql", log=say)
show_results(aud, ["Q-A1 base", "Q-A2 reconciliation with the legacy dataset", "Q-A3 admissions per patient", "Q-A3b summary"])
a1 = aud["Q-A1 base"].iloc[0]
decision("S4 · Decision D1: database", [
    ("base counts", f"{int(a1.n_admissions):,} admissions; {int(a1.n_patients):,} patients; "
                    f"{int(a1.n_delivery_hospitalizations):,} delivery hospitalizations",
     "revised tables of the obstetric cohort (the same as the maternal count audit)")])

## S4 · Count audit, group B — events and definition

How many events there are under each scope and tier, how they agree with the stillbirth codes of the delivery flag, how many have gestational weeks and where the code sits in the diagnosis list. The output ends with the table «Decisions D3 and D5».

| Result | What is done | In this model |
|---|---|---|
| 15 stillbirth codes fall outside the delivery flag, all with an abortive code | D3: population = delivery flag **or** stillbirth code | 19,440 admissions; 142 events, 129 strict |
| Gestational weeks exist only in ICD-10; 19% of the strict events with weeks are under 22 | D5: strict code at any gestational age; 22 and 28 weeks as sensitivity (ICD-10 only) | Main label: 129 strict events |

In [ ]:
# Group B: events and definition
show_results(aud, ["Q-B1 events by tier and ICD version", "Q-B2 events by scope", "Q-B3 label vs has_stillbirth_code",
                   "Q-B3b stillbirth code without delivery flag", "Q-B4 gestational weeks of events",
                   "Q-B4b weeks by delivery flag and abortive code", "Q-B5 position of the fetal-death code",
                   "Q-B6 strict codes outside the universe", "Q-B7 overlap with SMM and maternal death",
                   "Q-B8 events with abortive or ectopic codes"])
decision("S4 · Decisions D3 and D5", [
    ("stillbirth codes without the delivery flag (Q-B3b)", "all carry an abortive, ectopic or molar code",
     "D3: population = delivery flag or stillbirth code (19,440 admissions)"),
    ("strict codes outside the obstetric universe (Q-B6)", "none", "the universe does not lose events"),
    ("gestational weeks (Q-B4)", "only in ICD-10 admissions; some strict events under 22 weeks",
     "D5: strict code at any gestational age; 22 and 28 weeks as sensitivity (ICD-10 only)"),
    ("other outcomes in the same admission (Q-B7)", "severe maternal morbidity in some events; no maternal death",
     "no change: they are not part of the label")])

## S4 · Count audit, group C — patient overlap and same pregnancy (decision D4)

How many admissions each patient has, how far apart they are and how many events each unit of analysis keeps. It decides the record and the grouping of the validation.

| Result | What is done | In this model |
|---|---|---|
| 45% of the patients have more than one admission and 40% of the events are in a later admission | D4: one record per delivery hospitalization; every split grouped by patient | Only the first admission would lose 40% of the events |
| Pregnancy chains change the events by 5 at most | The 90-day pregnancy chain stays as sensitivity | 150, 149 and 149 events with chains of 30, 90 and 180 days |

In [ ]:
# Group C: patient overlap and same pregnancy
show_results(aud, ["Q-C1 event admissions per patient", "Q-C2 gap to the previous admission",
                   "Q-C3 gap to an earlier fetal-death admission", "Q-C3b event admissions by rank",
                   "Q-C4 events by unit of analysis"])
c4 = aud["Q-C4 events by unit of analysis"].set_index("unit")
first = c4.loc["First admission per patient", "n_units_with_event"]
alladm = c4.loc["Admission (any obstetric)", "n_units_with_event"]
decision("S4 · Decision D4: unit of analysis", [
    ("events kept by the first admission only", f"{int(first)} of {int(alladm)} ({100 * (1 - first / alladm):.0f}% lost)",
     "not used: one record per delivery hospitalization"),
    ("patients with more than one admission", "frequent (Q-A3b)", "every validation split is grouped by patient"),
    ("pregnancy chains of 30, 90 and 180 days", ", ".join(str(int(v)) for v in c4.filter(like="Pregnancy chain", axis=0)["n_units_with_event"]) + " events",
     "the 90-day chain is a sensitivity analysis")])

## S4 · Count audit, group D — era and coding

ICD-9 covers about 2008–2014 and ICD-10 about 2014–2021, so coding version and calendar period are almost confounded. The cell also computes the chi-square test for trend of the event rate in deliveries across the anchor-year groups (Cochran-Armitage), which the count audit left to be reproduced in code.

| Result | What is done | In this model |
|---|---|---|
| The rate rises with the period and the coding version moves with it | Recorded before any model: criterion C4 (stability over time) applies and calibration drift is expected | From 5.2 to 12.9 per 1,000 deliveries |
| Gestational weeks only in ICD-10 | The 22 and 28 week definitions are applied to ICD-10 admissions only | 45.5% of the deliveries have weeks |

In [ ]:
# Group D: era and coding
show_results(aud, ["Q-D1 events by ICD version", "Q-D1b events by anchor-year group", "Q-D1c approximate year by ICD version",
                   "Q-D2 gestational weeks by ICD version", "Q-D3 rates per 1,000 deliveries"])
# Chi-square test for trend (Cochran-Armitage) of the event rate in deliveries across the anchor-year groups (Q-D1b)
from scipy.stats import norm
d1b = aud["Q-D1b events by anchor-year group"].dropna(subset=["anchor_year_group"])
d1b = d1b[d1b["n_deliveries"] > 0].sort_values("anchor_year_group")
x = np.arange(len(d1b), dtype=float)
nd, ev = d1b["n_deliveries"].to_numpy(float), d1b["n_events_in_deliveries"].to_numpy(float)
p_bar = ev.sum() / nd.sum()
z_trend = (x * (ev - nd * p_bar)).sum() / np.sqrt(p_bar * (1 - p_bar) * ((nd * x ** 2).sum() - (nd * x).sum() ** 2 / nd.sum()))
p_trend = 2 * norm.sf(abs(z_trend))
show("S4 · Test for trend of the event rate in deliveries (Cochran-Armitage)", pd.DataFrame([{
    "groups": len(d1b), "first group, per 1,000": round(1000 * ev[0] / nd[0], 2),
    "last group, per 1,000": round(1000 * ev[-1] / nd[-1], 2), "z": round(z_trend, 2), "p": round(p_trend, 4)}]))
decision("S4 · Era and coding", [
    ("event rate by period", f"from {1000 * ev[0] / nd[0]:.1f} to {1000 * ev[-1] / nd[-1]:.1f} per 1,000 deliveries (p = {p_trend:.3f})",
     "criterion C4 applies; calibration drift is expected; reported as a limitation"),
    ("ICD version and period", "almost confounded (Q-D1c)", "no sensitivity analysis by version can separate them"),
    ("gestational weeks", "only in ICD-10 admissions (Q-D2)", "22 and 28 week definitions only in ICD-10")])

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 02
notebook_summary([
    ("obstetric admissions / patients", lambda: f"{int(a1.n_admissions):,} / {int(a1.n_patients):,}"),
    ("delivery hospitalizations", lambda: f"{int(a1.n_delivery_hospitalizations):,}"),
    ("events kept by the first admission only", lambda: f"{int(first)} of {int(alladm)}"),
    ("event rate in deliveries, first and last period", lambda: f"{1000 * ev[0] / nd[0]:.2f} and {1000 * ev[-1] / nd[-1]:.2f} per 1,000"),
    ("test for trend, p", lambda: round(p_trend, 4)),
    ("decisions settled", "D1 database, D3 population, D4 unit, D5 outcome"),
    ("statements of the script that failed", lambda: ", ".join(e[0] for e in aud["errors"]) or "none"),
    ("next step", "unit of analysis (S5) and prediction point (S6), notebook 03"),
])
write_outputs(OUT_FILE, "Notebook 02 (S4), fetal death MIMIC-IV (M3); aggregate output only")